## AGU figure guidelines

From the AGU journal graphic requirements, https://www.agu.org/Publish-with-AGU/Publish/Author-Resources/Graphic-Requirements

- Size
    - One column: 50 to 85 mm wide (1.97 to 3.35 in)
    - Two columns: 105 to 170 mm wide (4.13 to 6.69 in)
    - Height: 228 mm at most (8.98 in)
- Text
    - Font: Arial, Helvetica, Times, or Symbol, or the text is outlined or embedded
    - Size: close to 8 pt at print size, 6 pt for subscripts and superscripts
- Lines: 0.5 pt or thicker, no hairlines
- Raster resolution: 300 to 600 ppi at print size
- Color
    - RGB mode
    - Readable with a color vision deficiency: no rainbow and no red-green palettes
    - Do not rely on color alone: pair it with line styles, symbols, or patterns
- File format: JPG, TIFF, EPS, PS, or PDF
- Submission
    - First submission: figures inline with the text and caption
    - Revision: each main-text figure as its own file, supplement figures inline with their captions

In [ ]:
from pathlib import Path
from typing import cast
import sys

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.config import load_config, resolve_data_dir
from eddy_tracking.preprocess.streamline import trace_mean_streamline

EXPERIMENT = 'gulf_stream_20240305_20260531'
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_config(EXPERIMENT)

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

In [ ]:
from datetime import date

from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.patches import Rectangle
from eddy_tracking.preprocess.swot import index_swot_files_by_date

REGION_DATE = date(2025, 1, 24)
lon_range = cfg['base']['region']['lon_range']
lat_range = cfg['base']['region']['lat_range']

swot_files = sorted(resolve_data_dir(cfg, 'swot_dir').glob('*.nc'))
swot_by_date = index_swot_files_by_date(resolve_data_dir(cfg, 'swot_dir'))
with xr.open_dataset(swot_by_date[REGION_DATE]) as ds:
    region_adt = ds['adt_full'].to_numpy()[0]
    lon = ds['longitude'].to_numpy()
    lat = ds['latitude'].to_numpy()

# Trace the GS axis on the time-mean ADT field, at the level with the fastest mean flow along it
streamline = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/streamline.parquet')
mean_axis = trace_mean_streamline(swot_files, tuple(cfg['gulf_stream']['adt_level_range']))
axis_lon = mean_axis.lon
axis_lat = mean_axis.lat

fig = plt.figure(figsize=(6.69, 3.7))
ax = cast(GeoAxes, fig.add_axes((0.07, 0.085, 0.81, 0.9), projection=ccrs.PlateCarree()))
ax.set_extent([*lon_range, *lat_range], crs=ccrs.PlateCarree())
adt_fill = ax.contourf(lon, lat, region_adt, levels=np.arange(-1.25, 1.2501, 0.05), cmap='RdBu_r', transform=ccrs.PlateCarree(), zorder=1)
ax.add_feature(cfeature.LAND.with_scale('10m'), facecolor='#e8e8e8', zorder=2)
ax.coastlines(resolution='10m', color='#4d4d4d', linewidth=0.5, zorder=3)
for name, name_lon, name_lat in (('Gulf Stream', -70, 38.5), ('Sargasso Sea', -62, 30.5)):
    ax.text(name_lon, name_lat, name, fontsize=9, fontstyle='italic', color='#262626', ha='center', va='center', transform=ccrs.PlateCarree(), zorder=5)
gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='#808080', alpha=0.5, xlocs=range(-80, -55, 5), ylocs=range(30, 45, 2), zorder=4)
gridlines.top_labels = False
gridlines.right_labels = False
gridlines.xlabel_style = {'size': 8}
gridlines.ylabel_style = {'size': 8}
adt_bar = fig.colorbar(adt_fill, cax=ax.inset_axes((1.02, 0.125, 0.025, 0.75)))
adt_bar.set_ticks([-1, -0.5, 0, 0.5, 1], labels=['−1', '−0.5', '0', '0.5', '1'])
adt_bar.set_label('Absolute dynamic topography (m)')
adt_bar.outline.set_linewidth(0.6)
inset = cast(GeoAxes, ax.inset_axes((0.03, 0.68, 0.24, 0.3), projection=ccrs.PlateCarree()))
inset.set_extent([-100, -30, 10, 55], crs=ccrs.PlateCarree())
inset.add_feature(cfeature.LAND.with_scale('110m'), facecolor='#e8e8e8', edgecolor='#808080', linewidth=0.3)
inset.add_feature(cfeature.OCEAN.with_scale('110m'), facecolor='#dbe9f6')
inset.add_patch(Rectangle((lon_range[0], lat_range[0]), lon_range[1] - lon_range[0], lat_range[1] - lat_range[0], linewidth=1, edgecolor='red', facecolor='red', alpha=0.25, transform=ccrs.PlateCarree()))
plt.show()

Figure 1. Absolute dynamic topography across the study region (81°W to 56°W, 29°N to 44°N) on 2025-01-24.

Goal: Display the region that we're focusing on

In [ ]:
from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from eddy_tracking.preprocess.swot import index_swot_files_by_date
from eddy_tracking.preprocess.tracks import load_track_observations

NEAR_AXIS_KM = 150
CONTOUR_EVERY_DAYS = 10
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
age_cmaps = {'cyclone': ListedColormap(plt.get_cmap('Blues')(np.linspace(0.3, 1, 256))), 'anticyclone': ListedColormap(plt.get_cmap('Reds')(np.linspace(0.3, 1, 256)))}

eddy_tracks = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['crossed_axis'] = eddy_tracks['movement'].eq(target_class)
eddy_tracks['is_target'] = eddy_tracks['crossed_axis'] | (eddy_tracks['birth_distance_km'].abs().le(NEAR_AXIS_KM) & eddy_tracks['death_side'].eq(target_class.str[1]))
eddy_tracks['lifetime_days'] = (eddy_tracks['death_date'] - eddy_tracks['birth_date']).dt.days
examples = eddy_tracks.loc[eddy_tracks['crossed_axis']].sort_values('lifetime_days', ascending=False).drop_duplicates('polarity').set_index('polarity')
track_observations = load_track_observations(EXPERIMENT)
example_tracks = {
    polarity: cast(pd.DataFrame, track_observations.loc[track_observations['polarity'].eq(polarity) & track_observations['track_id'].eq(int(examples.loc[polarity, 'track_id']))]).sort_values('date')
    for polarity in polarity_colors
}
shared_dates = np.intersect1d(example_tracks['cyclone']['date'], example_tracks['anticyclone']['date'])
snapshot_date = cast(pd.Timestamp, pd.Timestamp(shared_dates[len(shared_dates) // 2]))
swot_by_date = index_swot_files_by_date(resolve_data_dir(cfg, 'swot_dir'))
with xr.open_dataset(swot_by_date[snapshot_date.date()]) as ds:
    sla = ds['sla'].to_numpy()[0]
    sla_lon = ds['longitude'].to_numpy()
    sla_lat = ds['latitude'].to_numpy()

lon_range = cfg['base']['region']['lon_range']
lat_range = cfg['base']['region']['lat_range']
all_lon = np.concatenate([np.concatenate(track['contour_lon'].to_list()) for track in example_tracks.values()])
all_lat = np.concatenate([np.concatenate(track['contour_lat'].to_list()) for track in example_tracks.values()])
# Panel (b) keeps the aspect ratio of panel (a), so the two maps are the same height side by side
map_aspect = (lat_range[1] - lat_range[0]) / (lon_range[1] - lon_range[0])
example_lon = (np.floor(all_lon.min()) - 1, np.ceil(all_lon.max()) + 1)
example_mid_lat = (all_lat.min() + all_lat.max()) / 2 - 0.5
example_half_lat = (example_lon[1] - example_lon[0]) * map_aspect / 2
example_extent = (*example_lon, example_mid_lat - example_half_lat, example_mid_lat + example_half_lat)

fig = plt.figure(figsize=(6.69, 3.05))
maps = fig.add_gridspec(1, 2, left=0.065, right=0.99, bottom=0.25, top=0.93, wspace=0.19)
snapshot_ax = cast(GeoAxes, fig.add_subplot(maps[0], projection=ccrs.PlateCarree()))
snapshot_ax.set_extent([*lon_range, *lat_range], crs=ccrs.PlateCarree())
sla_mesh = snapshot_ax.pcolormesh(sla_lon, sla_lat, sla, cmap='RdBu_r', vmin=-0.8, vmax=0.8, alpha=0.8, shading='nearest', rasterized=True, transform=ccrs.PlateCarree(), zorder=1)
sla_bar = fig.colorbar(sla_mesh, cax=snapshot_ax.inset_axes((0, -0.2, 1, 0.045)), orientation='horizontal')
sla_bar.set_label('Sea level anomaly (m)')
sla_bar.set_ticks([-0.8, -0.4, 0, 0.4, 0.8], labels=['−0.8', '−0.4', '0', '0.4', '0.8'])
sla_bar.ax.tick_params(length=2)
sla_bar.outline.set_linewidth(0.6)
axis = streamline.loc[streamline['date'].eq(snapshot_date)].sort_values('point_idx')
snapshot_ax.plot(axis['lon'], axis['lat'], color='black', linewidth=1.0, transform=ccrs.PlateCarree(), zorder=4)
today = track_observations.loc[track_observations['date'].eq(snapshot_date)].merge(eddy_tracks[['polarity', 'track_id', 'is_target']], on=['polarity', 'track_id'])
for polarity, contour_lon, contour_lat, is_target in zip(today['polarity'], today['contour_lon'], today['contour_lat'], today['is_target']):
    snapshot_ax.plot(np.append(contour_lon, contour_lon[0]), np.append(contour_lat, contour_lat[0]), color=polarity_colors[polarity], linewidth=1.4 if is_target else 0.7, linestyle='solid' if is_target else (0, (3, 1.5)), transform=ccrs.PlateCarree(), zorder=5)
snapshot_ax.set_title(f'$\\bf{{(a)}}$ Tracked eddies on {snapshot_date:%Y-%m-%d}', loc='left')
snapshot_ax.legend(handles=[
    Line2D([], [], color=polarity_colors['cyclone'], linewidth=1.4, label='Cyclone'),
    Line2D([], [], color=polarity_colors['anticyclone'], linewidth=1.4, label='Anticyclone'),
    Line2D([], [], color='#444444', linewidth=1.4, label='Target eddy'),
    Line2D([], [], color='#444444', linewidth=0.7, linestyle=(0, (3, 1.5)), label='Nontarget eddy'),
    Line2D([], [], color='black', linewidth=1.0, label='Gulf Stream axis'),
], loc='lower right', frameon=True, framealpha=0.9, edgecolor='none', fontsize=6.5, handlelength=1.8, borderpad=0.3, labelspacing=0.15)

# Panel (b): both example tracks on one map, each contour every CONTOUR_EVERY_DAYS days, colored by age
example_ax = cast(GeoAxes, fig.add_subplot(maps[1], projection=ccrs.PlateCarree()))
example_ax.set_extent(example_extent, crs=ccrs.PlateCarree())
example_ax.plot(axis_lon, axis_lat, color='black', linewidth=1.0, linestyle='--', transform=ccrs.PlateCarree(), zorder=4)
for panel, (polarity, track) in enumerate(example_tracks.items()):
    first_date, last_date = track['date'].iloc[0], track['date'].iloc[-1]
    age_days = (track['date'] - first_date).dt.days
    age_norm = Normalize(0, int(age_days.iloc[-1]))
    drawn = track.loc[age_days.mod(CONTOUR_EVERY_DAYS).eq(0) | track['date'].eq(last_date)]
    for contour_lon, contour_lat, age in zip(drawn['contour_lon'], drawn['contour_lat'], age_days.loc[drawn.index]):
        example_ax.plot(np.append(contour_lon, contour_lon[0]), np.append(contour_lat, contour_lat[0]), color=age_cmaps[polarity](age_norm(age)), linewidth=0.7, transform=ccrs.PlateCarree(), zorder=5)
    age_bar = fig.colorbar(ScalarMappable(norm=age_norm, cmap=age_cmaps[polarity]), cax=example_ax.inset_axes((0.53 * panel, -0.2, 0.47, 0.045)), orientation='horizontal')
    age_bar.set_label(f'{polarity.capitalize()} age (days)')
    age_bar.ax.xaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 5, 10]))
    age_bar.ax.tick_params(length=2)
    age_bar.outline.set_linewidth(0.6)
    here = today.loc[today['polarity'].eq(polarity) & today['track_id'].eq(int(examples.loc[polarity, 'track_id']))]
    for contour_lon, contour_lat in zip(here['contour_lon'], here['contour_lat']):
        snapshot_ax.text(contour_lon.max() + 0.2, contour_lat.max(), 'b', fontweight='bold', transform=ccrs.PlateCarree(), zorder=6)
example_ax.set_title('$\\bf{(b)}$ Example target eddy tracks', loc='left')
example_ax.legend(handles=[Line2D([], [], color='black', linewidth=1.0, linestyle='--', label='Mean Gulf Stream axis')], loc='lower right', frameon=True, framealpha=0.9, edgecolor='none', fontsize=6.5, handlelength=1.8, borderpad=0.3)
for ax in (snapshot_ax, example_ax):
    ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=2)
    ax.coastlines(resolution='50m', color='#666666', linewidth=0.5, zorder=3)
    gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='#bbbbbb', xlocs=range(-80, -55, 5), ylocs=range(30, 45, 2), zorder=3)
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.xlabel_style = {'size': 8}
    gridlines.ylabel_style = {'size': 8}
print(eddy_tracks.loc[eddy_tracks['is_target']].groupby('polarity').size().rename('target eddies').to_string())
plt.show()

Figure 2. (a) Eddies tracked on 2025-01-24 over sea level anomaly; target eddies (31 cyclones, 32 anticyclones) are outlined in bold, nontarget eddies dashed. (b) Tracks of one target cyclone and one target anticyclone, drawn every 10 days and colored by days since first detection.

Goal: Display the eddy tracking and contour identification, which types of eddies the study targets, and an example of eddy track progression over an eddy's lifetime.

In [ ]:
from matplotlib.ticker import MaxNLocator

N_AGE_BINS = 5
N_RADIAL_BINS = 10
MAX_RADIUS = 2
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
identity_columns = ['polarity', 'track_id']
shown_pigments = {
    'Tchla': 'Total chlorophyll-a', 'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
    'ButFuco': "19'-But-fucoxanthin", 'HexFuco': "19'-Hex-fucoxanthin", 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin', 'Fuco': 'Fucoxanthin',
    'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Perid': 'Peridinin',
}
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
radial_edges = np.linspace(0, MAX_RADIUS, N_RADIAL_BINS + 1)

plankton_table = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet')
pigment_table = pd.read_parquet(DATA_DIR / 'gold/eddy_pigment_table.parquet')
pigment_table['polarity'] = cast(pd.Series, pigment_table['polarity']).map({0: 'anticyclone', 1: 'cyclone'})
plankton_rings = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_rings.parquet')


def draw_lifetime_figure(eddies: pd.DataFrame, labels: dict[str, str]) -> None:
    plankton = plankton_table.merge(eddies, on=identity_columns)
    pigments = pigment_table.merge(eddies, on=identity_columns)
    analysis = pd.concat([
        plankton[identity_columns + ['date', 'age_frac']].assign(source='chl', variable='CHL', concentration=plankton['CHL']),
        pigments[identity_columns + ['date', 'age_frac']].assign(source='sdp').join(
            cast(pd.DataFrame, pigments[[f'eddy_mean_{pigment}' for pigment in shown_pigments]]).rename(columns={f'eddy_mean_{pigment}': pigment for pigment in shown_pigments}),
        ).melt(id_vars=identity_columns + ['date', 'age_frac', 'source'], var_name='variable', value_name='concentration'),
    ], ignore_index=True).sort_values(identity_columns + ['variable', 'date'])
    analysis['age_bin'] = np.minimum((analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
    analysis['change'] = analysis['concentration'] - analysis.groupby(identity_columns + ['variable'])['concentration'].transform('first')
    eddy_bins = analysis.groupby(['source', 'variable'] + identity_columns + ['age_bin'])['change'].mean().reset_index()
    n_eddies = analysis.groupby(['source', 'polarity'])['track_id'].nunique()
    rng = np.random.default_rng(RANDOM_SEED)
    summary_rows = []
    for keys, source_bins in eddy_bins.groupby(['source', 'polarity']):
        source, polarity = cast(tuple[str, str], keys)
        eddy_ids = sorted(source_bins['track_id'].unique())
        draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
        for variable, variable_bins in source_bins.groupby('variable'):
            matrix = variable_bins.pivot(index='track_id', columns='age_bin', values='change').reindex(index=eddy_ids, columns=range(N_AGE_BINS)).to_numpy(dtype=float)
            counts = np.isfinite(matrix).sum(axis=0)
            means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(N_AGE_BINS, np.nan), where=counts > 0)
            sampled = matrix[draws]
            sampled_counts = np.isfinite(sampled).sum(axis=1)
            sampled_means = np.divide(np.nansum(sampled, axis=1), sampled_counts, out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0)
            low, high = np.nanquantile(sampled_means, [0.025, 0.975], axis=0)
            summary_rows.append(pd.DataFrame({
                'source': source, 'polarity': polarity, 'variable': variable, 'age_bin': range(N_AGE_BINS),
                'age_midpoint': bin_centers, 'mean': means, 'ci_low': low, 'ci_high': high, 'n_eddies': counts,
            }))
    lifetime_summary = pd.concat(summary_rows, ignore_index=True)

    radial = plankton_rings.merge(cast(pd.DataFrame, analysis.loc[analysis['variable'].eq('CHL'), identity_columns + ['date', 'age_bin']]), on=identity_columns + ['date'])
    radial_grid = radial.groupby(identity_columns + ['age_bin', 'radial_bin'])['CHL'].mean().groupby(['polarity', 'age_bin', 'radial_bin']).mean()
    chl_norm = Normalize(np.floor(radial_grid.min() / 0.02) * 0.02, np.ceil(radial_grid.max() / 0.02) * 0.02)

    def draw_change(ax: Axes, source: str, variable: str) -> None:
        ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
        for polarity, color in polarity_colors.items():
            result = lifetime_summary.loc[lifetime_summary['polarity'].eq(polarity) & lifetime_summary['variable'].eq(variable)].sort_values('age_bin')
            ax.errorbar(result['age_midpoint'], result['mean'], yerr=[result['mean'] - result['ci_low'], result['ci_high'] - result['mean']], fmt='none', ecolor=color, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=2)
            ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, linewidth=1.2, markersize=3, markeredgecolor='white', markeredgewidth=0.5, label=f'{labels[polarity]} (n = {n_eddies[source, polarity]})', zorder=3)
        ax.xaxis.set_ticks(np.linspace(0, 1, 6))
        ax.set_xlim(0, 1)
        ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
        ax.set_axisbelow(True)
        locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
        ax.yaxis.set_major_locator(locator)
        ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
        ax.yaxis.set_ticks(ticks)
        ax.set_ylim(ticks[0], ticks[-1])

    fig = plt.figure(figsize=(6.69, 8.9))
    rows = fig.add_gridspec(2, 1, left=0.1, right=0.9, bottom=0.05, top=0.965, height_ratios=(2.0, 8.0), hspace=0.18)
    top = rows[0].subgridspec(1, 2, width_ratios=(1.6, 1.45), wspace=0.35)
    chl_ax = cast(Axes, fig.add_subplot(top[0]))
    heat_axes = cast(np.ndarray, top[1].subgridspec(1, 2, wspace=0.2).subplots(sharey=True))
    pigment_axes = cast(np.ndarray, rows[1].subgridspec(5, 3, wspace=0.45, hspace=0.5).subplots(sharex=True))
    cast(Axes, pigment_axes.flat[len(shown_pigments) + 1]).remove()
    draw_change(chl_ax, 'chl', 'CHL')
    chl_ax.set_title('$\\bf{(a)}$ Change in CHL', loc='left', pad=12.4)
    chl_ax.set_ylabel('$\\Delta$CHL (mg m$^{-3}$)')
    chl_ax.set_xlabel('Fraction of observed track')
    chl_ax.legend(loc='lower left', handlelength=2.2, borderaxespad=0.2, labelspacing=0.3)
    heat_cmap = plt.get_cmap('viridis').copy()
    heat_cmap.set_bad('#e6e6e6')
    for ax, polarity in zip(heat_axes, polarity_colors):
        ax = cast(Axes, ax)
        grid = radial_grid.loc[polarity].unstack('radial_bin').reindex(index=range(N_AGE_BINS), columns=range(N_RADIAL_BINS)).to_numpy(dtype=float).T
        ax.pcolormesh(bin_edges, radial_edges, np.ma.masked_invalid(grid), cmap=heat_cmap, norm=chl_norm, edgecolors='white', linewidth=0.5)
        ax.axhline(1, color='#222222', linewidth=0.8, linestyle=(0, (4, 2.5)), zorder=3)
        ax.set_aspect('equal')
        ax.set_title(f'{polarity.capitalize()}s', pad=2)
        ax.xaxis.set_ticks([0, 0.5, 1], ['0', '0.5', '1'])
        ax.xaxis.set_ticks(bin_edges, minor=True)
        ax.yaxis.set_ticks([0, 0.5, 1, 1.5, 2], ['0', '0.5', '1', '1.5', '2'])
        ax.yaxis.set_ticks(radial_edges, minor=True)
        ax.tick_params(length=2)
        ax.tick_params(which='minor', length=1.2)
    cast(Axes, heat_axes[0]).set_ylabel('Distance from center (speed radii)')
    cast(Axes, heat_axes[0]).text(0, 1.1, '$\\bf{(b)}$ CHL by radius and age', transform=heat_axes[0].transAxes, va='bottom', ha='left')
    cast(Axes, heat_axes[0]).set_xlabel('Fraction of observed track')
    cast(Axes, heat_axes[0]).xaxis.set_label_coords(1.1, -0.12)
    heat_bar = fig.colorbar(ScalarMappable(norm=chl_norm, cmap=heat_cmap), cax=cast(Axes, heat_axes[1]).inset_axes((1.1, 0, 0.08, 1)))
    heat_bar.set_label('CHL (mg m$^{-3}$)')
    heat_bar.ax.tick_params(length=2)
    heat_bar.ax.yaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]))
    heat_bar.outline.set_linewidth(0.5)
    for ax, letter, (pigment, label) in zip(pigment_axes.flat, 'cdefghijklmno', shown_pigments.items()):
        ax = cast(Axes, ax)
        draw_change(ax, 'sdp', pigment)
        ax.set_title(f'$\\bf{{({letter})}}$ {label}', loc='left')
    for ax in (pigment_axes[-1, 0], *pigment_axes[-2, 1:]):
        ax = cast(Axes, ax)
        ax.tick_params(labelbottom=True)
        ax.set_xlabel('Fraction of observed track')
    legend_ax = cast(Axes, pigment_axes.flat[len(shown_pigments)])
    legend_ax.axis('off')
    legend_ax.legend(*cast(Axes, pigment_axes.flat[0]).get_legend_handles_labels(), loc='upper left')
    pigment_box = rows[1].get_position(fig)
    fig.text(0.008, (pigment_box.y0 + pigment_box.y1) / 2, 'Change from first observation (mg m$^{-3}$)', rotation=90, va='center', ha='left')
    plt.show()


draw_lifetime_figure(eddy_tracks.loc[eddy_tracks['is_target'], identity_columns], {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'})

In [ ]:
draw_lifetime_figure(eddy_tracks.loc[~eddy_tracks['is_target'], identity_columns], {'cyclone': 'Nontarget cyclones', 'anticyclone': 'Nontarget anticyclones'})

Figure 3.

Goal: Show that the interior CHL, pigments, and/or PFT classes change over an eddy's lifetime depending on where it moves

Figure 4.

Goal: Show how SST changes over time within eddies to become more similar to their surrounding waters